In [1]:
import os
import time
import threading
from datetime import datetime
from collections import deque

# =========================
# 설정
# =========================
WATCH_DIR = "/root"
LOG_FILE = "/var/log/jupyter/jupyter.log"

MAX_LOG_SIZE = 10 * 1024 * 1024   # 10MB
MAX_LINES = 1000
CHECK_INTERVAL = 2


# =========================
# 로그 로테이션 (용량 제한)
# =========================
def rotate_log_by_size():
    if os.path.exists(LOG_FILE):
        if os.path.getsize(LOG_FILE) > MAX_LOG_SIZE:
            os.rename(LOG_FILE, LOG_FILE + ".old")
            open(LOG_FILE, "w").close()


# =========================
# 로그 기록
# =========================
def write_log(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_message = f"[{timestamp}] {message}"

    print(log_message)

    rotate_log_by_size()

    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, "r") as f:
            lines = deque(f.readlines(), maxlen=MAX_LINES)
    else:
        lines = deque(maxlen=MAX_LINES)

    lines.append(log_message + "\n")

    with open(LOG_FILE, "w") as f:
        f.writelines(lines)


# =========================
# 기존 로그 출력 (핵심 추가)
# =========================
def show_previous_logs():
    if os.path.exists(LOG_FILE):
        print("\n===== PREVIOUS LOGS (LAST 50) =====\n")

        with open(LOG_FILE, "r") as f:
            lines = f.readlines()[-50:]

            for line in lines:
                print(line.strip())

        print("\n===================================\n")


# =========================
# 파일 상태 스냅샷
# =========================
def snapshot(directory):
    state = {}

    for root, dirs, files in os.walk(directory):
        for name in files:
            path = os.path.join(root, name)

            try:
                stat = os.stat(path)
                state[path] = stat.st_mtime
            except:
                continue

    return state


# =========================
# 파일 감시
# =========================
def watch_files():
    write_log("FILE WATCH START")

    old_state = snapshot(WATCH_DIR)

    while True:
        time.sleep(CHECK_INTERVAL)

        new_state = snapshot(WATCH_DIR)

        created = set(new_state) - set(old_state)
        deleted = set(old_state) - set(new_state)

        modified = {
            f for f in set(old_state) & set(new_state)
            if old_state[f] != new_state[f]
        }

        for f in created:
            write_log(f"FILE CREATED : {f}")

        for f in modified:
            write_log(f"FILE MODIFIED : {f}")

        for f in deleted:
            write_log(f"FILE DELETED : {f}")

        old_state = new_state


# =========================
# 메인 실행
# =========================
def run_check():

    write_log("=" * 50)
    write_log("FILE MONITOR START")
    write_log("=" * 50)

    # ⭐ 핵심 추가: 과거 로그 출력
    show_previous_logs()


# =========================
# 파일 감시 스레드 시작
# =========================
t = threading.Thread(target=watch_files, daemon=True)
t.start()


# =========================
# 실행
# =========================
run_check()

[2026-06-02 17:43:48] FILE WATCH START
[2026-06-02 17:43:48] ==================================================
[2026-06-02 17:43:48] FILE MONITOR START
[2026-06-02 17:43:48] ==================================================

===== PREVIOUS LOGS (LAST 50) =====

[2026-06-02 17:31:26] FILE MODIFIED : /root/eunsoo/.ipynb_checkpoints/6_1_systemlog-checkpoint.ipynb
[2026-06-02 17:31:26] FILE MODIFIED : /root/.local/share/jupyter/nbsignatures.db
[2026-06-02 17:31:26] FILE MODIFIED : /root/.local/share/jupyter/nbsignatures.db
[2026-06-02 17:31:26] FILE MODIFIED : /root/eunsoo/.ipynb_checkpoints/6_1_systemlog-checkpoint.ipynb
[2026-06-02 17:31:26] FILE MODIFIED : /root/eunsoo/6_1_systemlog.ipynb
[2026-06-02 17:31:27] FILE MODIFIED : /root/eunsoo/6_1_systemlog.ipynb
[2026-06-02 17:31:27] FILE MODIFIED : /root/eunsoo/.ipynb_checkpoints/6_1_systemlog-checkpoint.ipynb
[2026-06-02 17:31:27] FILE MODIFIED : /root/.local/share/jupyter/nbsignatures.db
[2026-06-02 17:31:27] FILE MODIFIED : /root/.loc